In [53]:
import pandas as pd
import re

def parse_theorem_file(file_path):
    with open(file_path, 'r') as f:
        content = f.read()
    
    theorem_blocks = re.split(r'\n(?=THEOREM:)', content.strip())
    
    theorems_data = []
    
    for block in theorem_blocks:
        if not block.strip():
            continue
            
        lines = block.strip().split('\n')
        
        theorem_name = None
        goal = None
        expansions = None
        goals = []
        steps = []
        
        for line in lines:
            line = line.strip()
            
            if line.startswith('THEOREM:'):
                theorem_name = line.replace('THEOREM:', '').strip()
            elif line.startswith('GOAL:') and goal is None:
                goal = line.replace('GOAL:', '').strip()
            elif line.startswith('EXPANSIONS:'):
                expansions = int(line.replace('EXPANSIONS:', '').strip())
            elif line.startswith('GOAL:') and goal is not None:
                goals.append(line.replace('GOAL:', '').strip())
            elif line.startswith('STEP:'):
                steps.append(line.replace('STEP:', '').strip())
        
        theorems_data.append({
            'theorem_name': theorem_name,
            'expansions': expansions,
            'goal': goal,
            'sub_goals': goals,
            'steps': steps
        })
    
    return pd.DataFrame(theorems_data)



In [54]:
dfs_seed_sets = (
    {
        "nb": parse_theorem_file('./baseline/longest_theorems_proved_seed1.txt'),
        "nm": parse_theorem_file('./modular/longest_theorems_proved_seed1.txt'),
        "rb": parse_theorem_file('./residual/baseline/longest_theorems_proved_seed1.txt'),
        "rm": parse_theorem_file('./residual/modular/longest_theorems_proved_seed1.txt'),
    },
    {
        "nb": parse_theorem_file('./baseline/longest_theorems_proved_seed2.txt'),
        "nm": parse_theorem_file('./modular/longest_theorems_proved_seed2.txt'),
        "rb": parse_theorem_file('./residual/baseline/longest_theorems_proved_seed2.txt'),
        "rm": parse_theorem_file('./residual/modular/longest_theorems_proved_seed2.txt'),
    },
    {
        "nb": parse_theorem_file('./baseline/longest_theorems_proved_seed3.txt'),
        "nm": parse_theorem_file('./modular/longest_theorems_proved_seed3.txt'),
        "rb": parse_theorem_file('./residual/baseline/longest_theorems_proved_seed3.txt'),
        "rm": parse_theorem_file('./residual/modular/longest_theorems_proved_seed3.txt'),
    },
)

dfs_models_sets = {
    "Normal Baseline":   {
        "prev0": parse_theorem_file("./baseline/longest_theorems_proved_old.txt"),
        **{f"seed{s+1}": dfs_seed_sets[s]["nb"] for s in range(3)},
    },
    "Normal Modular":   {
        "prev0": parse_theorem_file("./modular/longest_theorems_proved_old.txt"),
        **{f"seed{s+1}": dfs_seed_sets[s]["nm"] for s in range(3)},
    },
    "Residual Baseline":   {
        "prev0": parse_theorem_file("./residual/baseline/longest_theorems_proved_old.txt"),
        **{f"seed{s+1}": dfs_seed_sets[s]["rb"] for s in range(3)},
    },
    "Residual Modular":   {
        "prev0": parse_theorem_file("./residual/baseline/longest_theorems_proved_old.txt"),
        **{f"seed{s+1}": dfs_seed_sets[s]["rm"] for s in range(3)},
    },
}

In [55]:
from itertools import combinations

def print_set_info(dfs):
    names = {k: set(df['theorem_name']) for k, df in dfs.items()}

    for k, s in names.items():
        print(f"{k}: {len(s)} theorems")

    print("\nJaccard Similarity:")
    for a, b in combinations(names, 2):
        intersection = len(names[a] & names[b])
        union = len(names[a] | names[b])
        print(f"  {a} x {b}: {((intersection / union) * 100):.2f}% ({intersection}/{union})")

    common_all = set.intersection(*names.values())
    print(f"\nCommon to all {len(names)}: {len(common_all)}")

    union_all = set.union(*names.values())
    print(f"Union: {len(union_all)}")

In [56]:
for i, dfs in enumerate(dfs_seed_sets):
    print(f"SEED: {i}")
    print("---------------------")
    print_set_info(dfs)
    print("=====================")

SEED: 0
---------------------
nb: 137 theorems
nm: 118 theorems
rb: 148 theorems
rm: 127 theorems

Jaccard Similarity:
  nb x nm: 50.89% (86/169)
  nb x rb: 62.86% (110/175)
  nb x rm: 51.72% (90/174)
  nm x rb: 45.36% (83/183)
  nm x rm: 58.06% (90/155)
  rb x rm: 51.10% (93/182)

Common to all 4: 63
Union: 207
SEED: 1
---------------------
nb: 137 theorems
nm: 118 theorems
rb: 150 theorems
rm: 128 theorems

Jaccard Similarity:
  nb x nm: 47.40% (82/173)
  nb x rb: 58.56% (106/181)
  nb x rm: 48.04% (86/179)
  nm x rb: 49.72% (89/179)
  nm x rm: 55.70% (88/158)
  rb x rm: 48.66% (91/187)

Common to all 4: 61
Union: 214
SEED: 2
---------------------
nb: 143 theorems
nm: 119 theorems
rb: 141 theorems
rm: 131 theorems

Jaccard Similarity:
  nb x nm: 55.03% (93/169)
  nb x rb: 59.55% (106/178)
  nb x rm: 50.55% (92/182)
  nm x rb: 47.73% (84/176)
  nm x rm: 57.23% (91/159)
  rb x rm: 50.28% (91/181)

Common to all 4: 67
Union: 212


In [57]:
for seed, model in enumerate(dfs_models_sets.keys()):
    print(f"MODEL: {model}")
    print("---------------------")
    print_set_info(dfs_models_sets[model])
    print("=====================")

MODEL: Normal Baseline
---------------------
prev0: 128 theorems
seed1: 137 theorems
seed2: 137 theorems
seed3: 143 theorems

Jaccard Similarity:
  prev0 x seed1: 67.72% (107/158)
  prev0 x seed2: 67.72% (107/158)
  prev0 x seed3: 68.32% (110/161)
  seed1 x seed2: 72.33% (115/159)
  seed1 x seed3: 72.84% (118/162)
  seed2 x seed3: 70.73% (116/164)

Common to all 4: 94
Union: 180
MODEL: Normal Modular
---------------------
prev0: 129 theorems
seed1: 118 theorems
seed2: 118 theorems
seed3: 119 theorems

Jaccard Similarity:
  prev0 x seed1: 58.33% (91/156)
  prev0 x seed2: 57.32% (90/157)
  prev0 x seed3: 58.97% (92/156)
  seed1 x seed2: 93.44% (114/122)
  seed1 x seed3: 91.13% (113/124)
  seed2 x seed3: 91.13% (113/124)

Common to all 4: 88
Union: 161
MODEL: Residual Baseline
---------------------
prev0: 127 theorems
seed1: 148 theorems
seed2: 150 theorems
seed3: 141 theorems

Jaccard Similarity:
  prev0 x seed1: 59.88% (103/172)
  prev0 x seed2: 63.91% (108/169)
  prev0 x seed3: 66.46% 

In [58]:
# from venny4py.venny4py import venny4py

# venny4py(sets=names, out='.', ext='png', dpi=150)

In [59]:
# all_names = sorted(union_all)
# membership = pd.DataFrame(
#     {k: [n in names[k] for n in all_names] for k in names},
#     index=all_names
# )
# print(membership)

In [60]:
# collapsed = {
#     k: df.groupby('theorem_name', as_index=False)['expansions'].first().set_index('theorem_name')
#     for k, df in dfs.items()
# }

# expansions = pd.concat(
#     {k: c['expansions'] for k, c in collapsed.items()},
#     axis=1
# )
# expansions_common = expansions.loc[list(common_all)]
# print(expansions_common)

In [61]:
# winners = expansions_common.idxmin(axis=1)
# # print(winners)

# print(winners.value_counts())